# Скачивание аудио из Telegram-канала (через бота)

Запускайте ячейки по порядку кнопкой ▶ слева.

**Что нужно заранее:**
- Токен бота от @BotFather
- Бот добавлен администратором в ваш канал

**Как это работает:** бот видит только новые сообщения. Пока ячейка
«Запуск» работает, перешлите аудио в канал со своего телефона —
бот их поймает и скачает с названиями.

## 1. Установка

In [ ]:
!pip install -q telethon

## 2. Токен бота
Вставьте токен между кавычками и запустите ячейку.

In [ ]:
BOT_TOKEN = "ВСТАВЬТЕ_ТОКЕН_СЮДА"

# Публичные тестовые api_id/api_hash из документации Telethon.
# Для входа по токену бота этого достаточно.
API_ID = 149467
API_HASH = "f6dab3d6d47c5d4a59b9a2b9f29e57a9"

## 3. Запуск приёмника
Запустите ячейку — она будет работать и ловить аудио.
Теперь зайдите в свой канал и **перешлите туда аудио**.
Скачивание остановится автоматически, если 60 секунд не приходит
новых файлов (или нажмите ⏹ «Стоп» вверху).

In [ ]:
import asyncio, json, re, time
from pathlib import Path
from telethon import TelegramClient, events
from telethon.tl.types import DocumentAttributeAudio, DocumentAttributeFilename

OUT = Path('audio'); OUT.mkdir(exist_ok=True)
manifest_path = OUT / 'manifest.json'
manifest = json.loads(manifest_path.read_text('utf-8')) if manifest_path.exists() else []
used = {m['file'] for m in manifest}
last = {'t': time.time()}

def clean(s):
    s = re.sub(r'[\\/:*?"<>|\n\r\t]+', ' ', (s or '').strip())
    s = re.sub(r'\s+', ' ', s).strip()
    return s[:150] or 'audio'

def attrs(doc):
    a = fn = None
    for x in doc.attributes:
        if isinstance(x, DocumentAttributeAudio): a = x
        elif isinstance(x, DocumentAttributeFilename): fn = x.file_name
    return a, fn

client = TelegramClient('bot_session', API_ID, API_HASH)

async def handle(msg):
    doc = getattr(msg, 'document', None)
    if not doc: return
    a, fn = attrs(doc)
    if not (a or (doc.mime_type and doc.mime_type.startswith('audio'))): return
    if a and not a.voice and (a.title or a.performer):
        title = ' - '.join(p for p in [(a.performer or '').strip(), (a.title or '').strip()] if p)
    elif fn:
        title = Path(fn).stem
    else:
        title = (msg.message or '').splitlines()[0] if msg.message else f'audio_{msg.id}'
    ext = (Path(fn).suffix if fn and '.' in fn else ('.ogg' if a and a.voice else '.mp3'))
    name = clean(title) + ext; i = 2
    while name in used or (OUT / name).exists():
        name = f'{clean(title)} ({i}){ext}'; i += 1
    used.add(name)
    print('↓', name)
    await client.download_media(msg, file=str(OUT / name))
    manifest.append({'message_id': msg.id, 'title': title, 'file': name,
                     'duration_sec': getattr(a, 'duration', None),
                     'date': msg.date.isoformat() if msg.date else None})
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), 'utf-8')
    last['t'] = time.time()
    print('  всего:', len(manifest))

@client.on(events.NewMessage())
async def _(e): await handle(e.message)

async def main():
    await client.start(bot_token=BOT_TOKEN)
    me = await client.get_me()
    print('Бот запущен: @' + me.username)
    print('Перешлите аудио в канал. Жду новые сообщения...')
    last['t'] = time.time()
    while True:
        await asyncio.sleep(5)
        if time.time() - last['t'] > 60 and manifest:
            print('60 секунд без новых файлов — останавливаюсь.'); break

await main()
print('Готово. Файлов в папке audio:', len(list(OUT.glob('*'))) - (1 if manifest_path.exists() else 0))

## 4. Скачать всё себе одним архивом

In [ ]:
import shutil
shutil.make_archive('audio_export', 'zip', 'audio')
from google.colab import files
files.download('audio_export.zip')